In [1]:
:set -XRankNTypes
:set -XTypeOperators
:set -XFlexibleInstances
:set -XFlexibleContexts
:set -XInstanceSigs
:set -XScopedTypeVariables
:set -XTupleSections
import Data.Maybe (fromMaybe)
import Data.List (intercalate)
putStrLn "Setup done." 

Setup done.

# 🔍 Оптики в Haskell

**Оптики** — это компонуемые инструменты для доступа к данным и их изменения в иммутабельных структурах.

Ключевые виды:
- `Lens` — фокус на одном значении в произведении
- `Prism` — фокус на одном конструкторе суммы
- `Traversal` — обход множества значений
- `Iso` — изоморфизм как оптика

## Contents

| # | Topic | Key idea |
|---|-------|----------|
| 1 | Lens via getter/setter | concrete record |
| 2 | Van Laarhoven Lens | higher-rank types |
| 3 | Lens composition | function composition |
| 4 | Prism | sum type optic |
| 5 | Iso | isomorphism as optic |
| 6 | Traversal (van Laarhoven) | multiple values, Applicative |
| 7 | Optics hierarchy (sketch) | Iso > Lens/Prism > Traversal |
| 8 | `lens` library | Edward Kmett's implementation |
| 9 | Unification via profunctors | `forall p. C p => p a b -> p s t` |
| 10 | Profunctor Traversal: `Wander` | equivalent to van Laarhoven |
| 11 | Grate: `Closed` profunctor | functional containers, zipWith |
| 12 | All optics table | full hierarchy via profunctor constraints |

## 1️⃣ Линза через геттер/сеттер

**Линза** `Lens s t a b` фокусируется на одном значении:

```
s содержит a  →  можно извлечь a из s
t содержит b  →  можно заменить a на b, получив t из s
```

```haskell
data CLens s a = CLens { cget :: s -> a, cset :: s -> a -> s }
```

Законы линзы:
1. `cget (cset s a) = a`              (get-set: получаем то, что установили)
2. `cset s (cget s) = s`              (set-get: установка тем, что получили, ничего не меняет)
3. `cset (cset s a) b = cset s b`     (set-set: двойная установка = одна)

In [2]:
-- Concrete Lens as a record
data CLens s a = CLens { cget :: s -> a, cset :: s -> a -> s }

-- Smart constructor
clens :: (s -> a) -> (s -> a -> s) -> CLens s a
clens = CLens

-- Operations
cview :: CLens s a -> s -> a
cview l s = cget l s

cover :: CLens s a -> (a -> a) -> s -> s
cover l f s = cset l s (f (cget l s))

csetL :: CLens s a -> a -> s -> s
csetL l a s = cset l s a

-- Example: nested record
data Person = Person { personName :: String, personAge :: Int } deriving Show
data Company = Company { ceoOf :: Person, cname :: String } deriving Show

nameLens :: CLens Person String
nameLens = clens personName (\p n -> p { personName = n })

ageLens :: CLens Person Int
ageLens = clens personAge (\p a -> p { personAge = a })

ceoLens :: CLens Company Person
ceoLens = clens ceoOf (\c p -> c { ceoOf = p })

-- Manual composition
ceoNameLens :: CLens Company String
ceoNameLens = clens
  (cget nameLens . cget ceoLens)
  (\co n -> cover ceoLens (\p -> csetL nameLens n p) co)

-- Demo
let alice = Person "Alice" 35
let acme  = Company alice "Acme Corp"
putStrLn $ "CEO: " ++ cview ceoNameLens acme
let acme2 = csetL ceoNameLens "Bob" acme
putStrLn $ "After rename: " ++ show (ceoOf acme2)
let acme3 = cover (ceoLens) (cover ageLens (+1)) acme
putStrLn $ "After birthday: " ++ show (personAge (ceoOf acme3))

CEO: Alice

After rename: Person {personName = "Bob", personAge = 35}

After birthday: 36

## 2️⃣ Линза Ван Лааровена

Стандартное кодирование Haskell использует **типы высшего ранга**:

```haskell
type Lens s t a b = forall f. Functor f => (a -> f b) -> s -> f t
```

Гениальная идея: одна функция `(a -> f b) -> s -> f t` выполняет роль и геттера, и сеттера, в зависимости от выбора `f`:
- `f = Identity` → модификация (setter/over)
- `f = Const r` → доступ (getter/view)

In [3]:
:set -XRankNTypes

-- Identity functor (for 'over')
newtype Identity a = Identity { runIdentity :: a }

instance Functor Identity where
  fmap f (Identity a) = Identity (f a)

-- Const functor (for 'view')
newtype Const r a = Const { getConst :: r }

instance Functor (Const r) where
  fmap _ (Const r) = Const r

-- Van Laarhoven Lens type
type VLLens s t a b = forall f. Functor f => (a -> f b) -> s -> f t
type VLLens' s a    = VLLens s s a a

-- Build a lens
vllens :: (s -> a) -> (s -> b -> t) -> VLLens s t a b
vllens get set f s = fmap (set s) (f (get s))

-- view: use Const functor
vlview :: VLLens' s a -> s -> a
vlview l s = getConst (l Const s)

-- over: use Identity functor
vlover :: VLLens s t a b -> (a -> b) -> s -> t
vlover l f s = runIdentity (l (Identity . f) s)

-- set: special case of over
vlset :: VLLens s t a b -> b -> s -> t
vlset l b = vlover l (const b)

-- Example lenses
vlFst :: VLLens' (a, b) a
vlFst = vllens fst (\(_, b) a -> (a, b))

vlSnd :: VLLens' (a, b) b
vlSnd = vllens snd (\(a, _) b -> (a, b))

-- Test
let p = (10 :: Int, "hello")
putStrLn $ "vlview vlFst (10,hello) = " ++ show (vlview vlFst p)
putStrLn $ "vlview vlSnd (10,hello) = " ++ show (vlview vlSnd p)
putStrLn $ "vlset  vlFst 99 (10,hello) = " ++ show (vlset vlFst 99 p)
putStrLn $ "vlover vlSnd (++ !) (10,hello) = " ++ show (vlover vlSnd (++"!") p)

vlview vlFst (10,hello) = 10

vlview vlSnd (10,hello) = "hello"

vlset  vlFst 99 (10,hello) = (99,"hello")

vlover vlSnd (++ !) (10,hello) = (10,"hello!")

## 3️⃣ Композиция линз = Композиция функций

Магия кодирования Ван Лааровена:

```haskell
-- Функции составляются стандартным (.)
-- Это АВТОМАТИЧЕСКИ составляет линзы!
fstL . sndL :: Lens (a, (b, c)) (a, (b, d)) c d
```

Не нужен специальный оператор — обычная `(.)` работает для любой вложенности!

In [4]:
:set -XRankNTypes

-- Reuse VL lens machinery
newtype Identity a = Identity { runIdentity :: a }
instance Functor Identity where fmap f (Identity a) = Identity (f a)

newtype Const r a = Const { getConst :: r }
instance Functor (Const r) where fmap _ (Const r) = Const r

type VLLens' s a = forall f. Functor f => (a -> f a) -> s -> f s

vllens :: (s -> a) -> (s -> a -> s) -> VLLens' s a
vllens get set f s = fmap (set s) (f (get s))

vlview :: VLLens' s a -> s -> a
vlview l s = getConst (l Const s)

vlover :: VLLens' s a -> (a -> a) -> s -> s
vlover l f s = runIdentity (l (Identity . f) s)

vlset :: VLLens' s a -> a -> s -> s
vlset l a = vlover l (const a)

-- Nested data
data Address = Address { street :: String, city :: String } deriving Show
data Employee = Employee { empName :: String, empAddr :: Address } deriving Show

-- Lenses
addrLens :: VLLens' Employee Address
addrLens = vllens empAddr (\e a -> e { empAddr = a })

streetLens :: VLLens' Address String
streetLens = vllens street (\a s -> a { street = s })

cityLens :: VLLens' Address String
cityLens = vllens city (\a c -> a { city = c })

-- COMPOSITION with (.)
empStreetLens :: VLLens' Employee String
empStreetLens = addrLens . streetLens

empCityLens :: VLLens' Employee String
empCityLens = addrLens . cityLens

-- Test composition
let emp = Employee "Alice" (Address "123 Main St" "Springfield")
putStrLn $ "Street: " ++ vlview empStreetLens emp
putStrLn $ "City:   " ++ vlview empCityLens emp
let emp2 = vlset empStreetLens "456 Oak Ave" emp
putStrLn $ "After move: " ++ show (empAddr emp2)
-- Chain 3 levels deep
data Corp = Corp { headOffice :: Employee } deriving Show
hqLens :: VLLens' Corp Employee
hqLens = vllens headOffice (\c e -> c { headOffice = e })
let corp = Corp emp
putStrLn $ "HQ city: " ++ vlview (hqLens . addrLens . cityLens) corp

Street: 123 Main St

City:   Springfield

After move: Address {street = "456 Oak Ave", city = "Springfield"}

HQ city: Springfield

## 4️⃣ Законы линз

Корректная («lawful») линза подчиняется трём законам, связывающим `get` и `set`:

- **get-put:** `set s (get s) = s` — записать то, что прочитали, ничего не меняет;
- **put-get:** `get (set s a) = a` — прочитать записанное даёт записанное;
- **put-put:** `set (set s a) b = set s b` — последняя запись побеждает.

Линза, нарушающая хоть один закон (например, сеттер с нормализацией или побочным
эффектом), формально линзой не является — она «well-behaved» только при всех трёх.


In [5]:
-- S4: lens laws (concrete getter/setter)
:set -XScopedTypeVariables
data CLens s a = CLens { cget :: s -> a, cset :: s -> a -> s }

data P = P { px :: Int, py :: Int } deriving (Eq, Show)
lx :: CLens P Int
lx = CLens px (\p v -> p { px = v })

lawGetPut :: Eq s => CLens s a -> s -> Bool
lawGetPut l s = cset l s (cget l s) == s
lawPutGet :: Eq a => CLens s a -> s -> a -> Bool
lawPutGet l s a = cget l (cset l s a) == a
lawPutPut :: Eq s => CLens s a -> s -> a -> a -> Bool
lawPutPut l s a b = cset l (cset l s a) b == cset l s b

demoLaws :: IO ()
demoLaws = do
  let s = P 3 4
  putStrLn ("get-put: " ++ show (lawGetPut lx s))
  putStrLn ("put-get: " ++ show (lawPutGet lx s 9))
  putStrLn ("put-put: " ++ show (lawPutPut lx s 9 12))
demoLaws

get-put: True
put-get: True
put-put: True

## 4️⃣ Призма — оптика для типов-сумм

**Призма** фокусируется на одном конструкторе суммового типа:

```haskell
type Prism s t a b = forall p f. (Choice p, Applicative f) => p a (f b) -> p s (f t)
```

Конкретная версия:
- `preview :: Prism s t a b -> s -> Maybe a` (извлечение, может не совпасть)
- `review  :: Prism s t a b -> b -> t`       (конструирование)

Отличие от линзы: линза всегда находит значение, призма — только при совпадении конструктора.

In [6]:
:set -XRankNTypes

-- Concrete Prism
data CPrism s a = CPrism
  { previewP :: s -> Maybe a    -- try to extract
  , reviewP  :: a -> s          -- inject
  }

-- Operations
ppreview :: CPrism s a -> s -> Maybe a
ppreview = previewP

preview' :: CPrism s a -> s -> Either s a
preview' p s = maybe (Left s) Right (previewP p s)

previewOr :: CPrism s a -> a -> s -> a
previewOr p def s = maybe def id (previewP p s)

pover :: CPrism s a -> (a -> a) -> s -> s
pover p f s = case previewP p s of
  Nothing -> s
  Just a  -> reviewP p (f a)

-- Standard prisms
_Just :: CPrism (Maybe a) a
_Just = CPrism id Just

_Right :: CPrism (Either e a) a
_Right = CPrism (either (const Nothing) Just) Right

_Left :: CPrism (Either a e) a
_Left = CPrism (either Just (const Nothing)) Left

-- Test
let vals = [Just 1, Nothing, Just 3, Nothing, Just 5] :: [Maybe Int]
putStrLn $ "preview _Just (Just 42) = " ++ show (ppreview _Just (Just (42::Int)))
putStrLn $ "preview _Just Nothing = " ++ show (ppreview _Just (Nothing :: Maybe Int))
putStrLn $ "over _Just (*2) (Just 5) = " ++ show (pover _Just (*2) (Just (5::Int)))
putStrLn $ "over _Just (*2) Nothing  = " ++ show (pover _Just (*2) (Nothing :: Maybe Int))

-- Traversal via prism: map over all Just values
let modified = map (pover _Just (*10)) vals
putStrLn $ "map (over _Just *10) = " ++ show modified

preview _Just (Just 42) = Just 42

preview _Just Nothing = Nothing

over _Just (*2) (Just 5) = Just 10

over _Just (*2) Nothing  = Nothing

map (over _Just *10) = [Just 10,Nothing,Just 30,Nothing,Just 50]

In [7]:
-- S5b: prism review/re (build, not only preview)
data CPrism s a = CPrism { previewP :: s -> Maybe a, reviewP :: a -> s }

_JustP :: CPrism (Maybe a) a
_JustP = CPrism id Just

_LeftP :: CPrism (Either e a) e
_LeftP = CPrism (\x -> case x of { Left e -> Just e; Right _ -> Nothing }) Left

demoPrism :: IO ()
demoPrism = do
  print (previewP _JustP (Just (5::Int)))             -- Just 5
  print (previewP _JustP (Nothing :: Maybe Int))      -- Nothing
  print (reviewP _JustP (7::Int))                     -- Just 7   (build)
  print (previewP _LeftP (Left "e" :: Either String Int))   -- Just "e"
  print (reviewP _LeftP "boom" :: Either String Int)        -- Left "boom"
  -- prism law: preview . review = Just
  print (previewP _JustP (reviewP _JustP (1::Int)))   -- Just 1
demoPrism

Line 8: Use lambda-case
Found:
\ x
  -> case x of
       Left e -> Just e
       Right _ -> Nothing
Why not:
\case
  Left e -> Just e
  Right _ -> Nothing

Just 5
Nothing
Just 7
Just "e"
Left "boom"
Just 1

## 5️⃣ Изо — изоморфизмы как оптики

**Изо** `Iso s t a b` говорит: «s и a изоморфны»:

```haskell
type Iso s t a b = forall p f. (Profunctor p, Functor f) => p a (f b) -> p s (f t)
```

Если `s ≅ a`, то можно конвертировать туда и обратно без потерь.

Изо является и линзой (уникальный фокус), и призмой (всегда совпадает).

In [8]:
-- Concrete Iso
data CIso s a = CIso { forward :: s -> a, backward :: a -> s }

-- Operations
iview :: CIso s a -> s -> a
iview = forward

iover :: CIso s a -> (a -> a) -> s -> s
iover i f = backward i . f . forward i

iset :: CIso s a -> a -> s -> s
iset i a = const (backward i a)

-- from: flip an iso
from :: CIso s a -> CIso a s
from (CIso f b) = CIso b f

-- Example isos
newtype Dollars = Dollars { cents :: Int } deriving Show
newtype Euros   = Euros   { eurocents :: Int } deriving Show

dollarsCents :: CIso Dollars Int
dollarsCents = CIso cents Dollars

-- String <-> [Char] (trivial, but illustrative)
packed :: CIso String [Char]
packed = CIso id id  -- String IS [Char] in Haskell

-- Curry iso: (a,b)->c  <->  a->b->c
curryIso :: CIso ((a,b) -> c) (a -> b -> c)
curryIso = CIso curry uncurry

-- Test
let d = Dollars 1050
putStrLn $ "Dollars to cents: " ++ show (iview dollarsCents d)
putStrLn $ "Cents to Dollars: " ++ show (iset dollarsCents 2500 d)
putStrLn $ "over dollarsCents (*2) = " ++ show (iover dollarsCents (*2) d)

-- Use curryIso
let uncurriedAdd = uncurry (+) :: (Int,Int) -> Int
let curriedAdd   = iview curryIso uncurriedAdd
putStrLn $ "curryIso: curriedAdd 3 4 = " ++ show (curriedAdd 3 4)

Dollars to cents: 1050

Cents to Dollars: Dollars {cents = 2500}

over dollarsCents (*2) = Dollars {cents = 2100}

curryIso: curriedAdd 3 4 = 7

## 7️⃣ AffineTraversal (Optional): фокус 0-или-1

**AffineTraversal** (она же `Optional`) фокусируется ровно на **нуле или одном**
элементе. Это недостающее звено решётки: она слабее линзы (фокус может отсутствовать)
и слабее призмы (структура может быть составной), но сильнее обхода (не более одной цели).
Линза = всегда 1 цель; призма = 0/1 цель в сумме; AffineTraversal = 0/1 в любом `s`.


In [9]:
-- S7: AffineTraversal / Optional (0-or-1 focus)
data Optional s a = Optional { matchO :: s -> Maybe a, setO :: s -> a -> s }

headO :: Optional [a] a
headO = Optional (\xs -> case xs of { (x:_) -> Just x; [] -> Nothing })
                 (\xs v -> case xs of { (_:t) -> v:t; [] -> [] })

overO :: Optional s a -> (a -> a) -> s -> s
overO o f s = case matchO o s of { Just a -> setO o s (f a); Nothing -> s }

demoOpt :: IO ()
demoOpt = do
  print (matchO headO [10,20,30::Int])     -- Just 10
  print (matchO headO ([] :: [Int]))       -- Nothing
  print (overO headO (*100) [1,2,3::Int])  -- [100,2,3]
  print (overO headO (*100) ([] :: [Int])) -- []
demoOpt

Line 5: Use lambda-case
Found:
\ xs
  -> case xs of
       (x : _) -> Just x
       [] -> Nothing
Why not:
\case
  (x : _) -> Just x
  [] -> Nothing

Just 10
Nothing
[100,2,3]
[]

## 6️⃣ Обход: доступ к нескольким значениям

**Traversal** фокусируется на *нескольких* значениях:

```haskell
type Traversal s t a b = forall f. Applicative f => (a -> f b) -> s -> f t
```

Законы:
1. `t pure = pure`                   (нейтральность)
2. `fmap (t f) . t g = getCompose . t (Compose . fmap f . g)`  (слияние)

Пример: `traversed :: Traversable t => Traversal (t a) (t b) a b`

```haskell
-- Изменить все элементы списка
over traversed (+1) [1,2,3]  -- [2,3,4]
```

In [10]:
:set -XRankNTypes

-- Reuse Identity and Const
newtype Identity a = Identity { runIdentity :: a }
instance Functor Identity where fmap f (Identity a) = Identity (f a)
instance Applicative Identity where
  pure = Identity
  Identity f <*> Identity x = Identity (f x)

-- For Const r, we need Monoid r to be Applicative
newtype ConstApp r a = ConstApp { getConstApp :: r }
instance Functor (ConstApp r) where fmap _ (ConstApp r) = ConstApp r
instance Monoid r => Applicative (ConstApp r) where
  pure _ = ConstApp mempty
  ConstApp r1 <*> ConstApp r2 = ConstApp (r1 <> r2)

-- Traversal type
type Traversal' s a = forall f. Applicative f => (a -> f a) -> s -> f s

-- traverse for lists IS a Traversal
eachInList :: Traversal' [a] a
eachInList f xs = sequenceA (map f xs)

-- toListOf: collect all focuses
toListOf :: Traversal' s a -> s -> [a]
toListOf t s = getConstApp (t (\a -> ConstApp [a]) s)

-- overT: modify all focuses
overT :: Traversal' s a -> (a -> a) -> s -> s
overT t f s = runIdentity (t (Identity . f) s)

-- Traversal for pairs (focuses both elements if same type)
both :: Traversal' (a, a) a
both f (x, y) = (,) <$> f x <*> f y

-- Test
let xs = [1..5 :: Int]
putStrLn $ "toListOf eachInList [1..5] = " ++ show (toListOf eachInList xs)
putStrLn $ "overT eachInList (*2) [1..5] = " ++ show (overT eachInList (*2) xs)
putStrLn $ "overT both (+10) (3,7) = " ++ show (overT both (+10) (3::Int, 7))
putStrLn $ "toListOf both (3,7) = " ++ show (toListOf both (3::Int, 7))

-- Traversal composition: modify nested lists
let nested = [[1,2],[3,4],[5]] :: [[Int]]
putStrLn $ "overT (eachInList . eachInList) (*2) [[1,2],[3,4],[5]] = "
         ++ show (overT (eachInList . eachInList) (*2) nested)

toListOf eachInList [1..5] = [1,2,3,4,5]

overT eachInList (*2) [1..5] = [2,4,6,8,10]

overT both (+10) (3,7) = (13,17)

toListOf both (3,7) = [3,7]

overT (eachInList . eachInList) (*2) [[1,2],[3,4],[5]] = [[2,4],[6,8],[10]]

## 9️⃣ Fold, Getter, Setter

Не все оптики «двунаправленные». Три ограниченные оптики лежат в основании решётки:

- **Getter** `s -> a` — только чтение одного значения;
- **Fold** `s -> [a]` — только чтение нескольких значений;
- **Setter** `(a -> b) -> s -> t` — только запись (модификация).

Lens и Traversal сильнее: из них выводятся и чтение, и запись. Getter/Fold/Setter —
это «половинки», полезные, когда направление одно.


In [11]:
-- S9: Fold / Getter / Setter
type Getter s a     = s -> a
type Fold s a       = s -> [a]
type Setter s t a b = (a -> b) -> s -> t

toView :: Getter s a -> s -> a
toView g = g
foldOf :: Fold s a -> s -> [a]
foldOf f = f
overS :: Setter s t a b -> (a -> b) -> s -> t
overS s = s

demoFGS :: IO ()
demoFGS = do
  let g   = fst :: Getter (Int,Char) Int
      fld = (\(a,b) -> [a,b]) :: Fold (Int,Int) Int
      st  = map :: Setter [Int] [Int] Int Int
  print (toView g (7,'z'))       -- 7
  print (foldOf fld (3,4))        -- [3,4]
  print (overS st (+1) [1,2,3])   -- [2,3,4]
demoFGS

7
[3,4]
[2,3,4]

## 1️⃣0️⃣ Индексированные оптики

**Индексированная** оптика проносит сквозь обход позицию/ключ элемента. Это позволяет
действиям зависеть от индекса (как `itraverse`, `imap`): не просто «обойти все», а
«обойти все, зная, где находишься».


In [12]:
-- S10: indexed optics (traversal carrying an index)
itraverseList :: Applicative f => (Int -> a -> f b) -> [a] -> f [b]
itraverseList f = go 0
  where go _ []     = pure []
        go i (x:xs) = (:) <$> f i x <*> go (i+1) xs

ifoldList :: (Int -> a -> b) -> [a] -> [b]
ifoldList f xs = zipWith f [0..] xs

demoIdx :: IO ()
demoIdx = do
  print (ifoldList (\i x -> (i,x)) "abc")   -- [(0,'a'),(1,'b'),(2,'c')]
  r <- itraverseList (\i x -> do
                        putStrLn (show i ++ " -> " ++ show x)
                        return (x*10)) [4,5,6::Int]
  print r                                    -- [40,50,60]
demoIdx

Line 8: Eta reduce
Found:
ifoldList f xs = zipWith f [0 .. ] xs
Why not:
ifoldList f = zipWith f [0 .. ]Line 12: Use (,)
Found:
\ i x -> (i, x)
Why not:
(,)

[(0,'a'),(1,'b'),(2,'c')]
0 -> 4
1 -> 5
2 -> 6
[40,50,60]

## 1️⃣1️⃣ Контейнерные оптики: Ix, At, Each

Самые «рабочие лошадки» на практике:

- **Ix** — частичный доступ по индексу/ключу (модификация на месте, no-op если нет);
- **At** — фокус на `Maybe v`: позволяет **вставлять и удалять** по ключу;
- **Each** — обход всех элементов контейнера (частный Traversal).


In [13]:
-- S11: container optics Ix / At / Each (local impls)
ixList :: Int -> (a -> a) -> [a] -> [a]
ixList i f xs = [ if j == i then f x else x | (j,x) <- zip [0..] xs ]

atKey :: Eq k => k -> [(k,v)] -> Maybe v
atKey k = lookup k
setAt :: Eq k => k -> Maybe v -> [(k,v)] -> [(k,v)]
setAt k Nothing  m = filter ((/= k) . fst) m
setAt k (Just v) m = (k,v) : filter ((/= k) . fst) m

eachList :: (a -> b) -> [a] -> [b]
eachList = map

demoCont :: IO ()
demoCont = do
  print (ixList 1 (*100) [1,2,3::Int])                       -- [1,200,3]
  print (atKey "b" [("a",1),("b",2::Int)])                   -- Just 2
  print (setAt "c" (Just 9) [("a",1::Int)])                  -- [("c",9),("a",1)]
  print (setAt "a" Nothing  [("a",1),("b",2::Int)])          -- [("b",2)]
  print (eachList (+1) [10,20::Int])                         -- [11,21]
demoCont

Line 6: Eta reduce
Found:
atKey k = lookup k
Why not:
atKey = lookup

[1,200,3]
Just 2
[("c",9),("a",1)]
[("b",2)]
[11,21]

## 9️⃣ Унификация через профункторы

Из `Profunctors.ipynb`, все оптики унифицируются как:

```
type Optic p s t a b = p a b -> p s t
```

| Оптика | Ограничение на `p` |
|--------|-------------------|
| Iso | `Profunctor p` |
| Lens | `Strong p` |
| Prism | `Choice p` |
| Grate | `Closed p` |
| Traversal | `Traversing p` |
| Setter | `Mapping p` |
| Fold | `(Choice, Strong) p` |

### Композиция

В обоих кодированиях (Ван Лааровена и профункторном):

```haskell
optic1 :: Optic p Big Medium
optic2 :: Optic p Medium Small

optic1 . optic2 :: Optic p Big Small
```

Композиция оптик = **композиция функций** в обоих кодированиях.

Категориальный вывод: оптики — это морфизмы в подходящей категории, и композиция морфизмов даёт нам бесплатно.

In [14]:
:set -XRankNTypes
:set -XFlexibleInstances

-- Define Profunctor locally (not imported in this notebook)
class MyProfunctor p where
  dimap :: (a' -> a) -> (b -> b') -> p a b -> p a' b'

instance MyProfunctor (->) where
  dimap f g h = g . h . f

-- Identity functor for Wander instance
newtype MyId a = MyId { runMyId :: a }
instance Functor MyId where fmap f (MyId a) = MyId (f a)
instance Applicative MyId where
  pure = MyId
  MyId f <*> MyId a = MyId (f a)

-- Wander class (using MyProfunctor)
class MyProfunctor p => Wander p where
  wander :: (forall f. Applicative f => (a -> f b) -> s -> f t)
         -> p a b -> p s t

instance Wander (->) where
  wander trav f = runMyId . trav (MyId . f)

-- Profunctor Traversal type
type MyTraversal s t a b = forall p. Wander p => p a b -> p s t

-- both: traverse both elements of a pair
both :: MyTraversal (a,a) (b,b) a b
both = wander (\f (x,y) -> (,) <$> f x <*> f y)

-- Use with (->): modify both
overBoth :: (a -> b) -> (a,a) -> (b,b)
overBoth = both

print (overBoth (*2) (3,5))   -- (6,10)
print (overBoth show (1,2))   -- ("1","2")

(6,10)

("1","2")

In [15]:
:set -XRankNTypes
:set -XFlexibleInstances

newtype MyId a = MyId { runMyId :: a }
instance Functor MyId where fmap f (MyId a) = MyId (f a)
instance Applicative MyId where
  pure = MyId
  MyId f <*> MyId a = MyId (f a)

-- Wander: profunctor Traversal constraint (standalone)
class Wander p where
  wander :: (forall f. Applicative f => (a -> f b) -> s -> f t)
         -> p a b -> p s t

instance Wander (->) where
  wander trav f = runMyId . trav (MyId . f)

type MyTraversal s t a b = forall p. Wander p => p a b -> p s t

both :: MyTraversal (a,a) (b,b) a b
both = wander (\f (x,y) -> (,) <$> f x <*> f y)

overBoth :: (a -> b) -> (a,a) -> (b,b)
overBoth = both

print (overBoth (*2) (3,5))   -- (6,10)
print (overBoth show (1,2))   -- ("1","2")

(6,10)

("1","2")

---

## 11. Grate: optic for coexponentials

### Definition

```haskell
class Profunctor p => Closed p where
  closed :: p a b -> p (x -> a) (x -> b)

type Grate s t a b = forall p. Closed p => p a b -> p s t
```

`closed` wraps a profunctor under a function type -- the dual of `first`.

### What Grate focuses on

A Grate works on *functional containers* -- structures of the form `x -> a`.
Where a Lens gets/sets a single element and a Traversal visits multiple,
a Grate zips two structures together with a combining function.

```haskell
zipWithOf :: Grate s t a b -> (a -> a -> b) -> s -> s -> t
```

### Categorical view

`Closed` = Tambara module for the co-exponential action `F a = x -> a`.
Where `Strong` is for products `(a, c)` and `Choice` for sums `Either a c`,
`Closed` is for the function space `x -> a`.
These three cover all three basic type formers in cartesian closed categories.

In [16]:
:set -XRankNTypes

-- Profunctor base (local)
class MyProfunctor2 p where
  dimap2 :: (a' -> a) -> (b -> b') -> p a b -> p a' b'

instance MyProfunctor2 (->) where
  dimap2 f g h = g . h . f

-- Closed profunctor
class MyProfunctor2 p => Closed p where
  closed :: p a b -> p (x -> a) (x -> b)

-- (->) instance
instance Closed (->) where
  closed f g x = f (g x)

-- Grate type
type MyGrate s t a b = forall p. Closed p => p a b -> p s t

-- Focus on the return value of a function
_cod :: MyGrate (x -> a) (x -> b) a b
_cod = closed

-- Example: Reader as functional container
type Reader r a = r -> a

f1 :: Int -> Int
f1 x = x * 2

f2 :: Int -> Int
f2 x = x + 10

-- Combine two Reader functions pointwise
addReaders :: Reader r Int -> Reader r Int -> Reader r Int
addReaders r1 r2 x = r1 x + r2 x

combined :: Int -> Int
combined = addReaders f1 f2

print (map combined [1..5])  -- [13,14,16,18,20]
putStrLn "Grate / Closed: OK"

[13,16,19,22,25]

Grate / Closed: OK

---

## 12. All optics via profunctor interface

Every optic is a natural transformation `forall p. Constraint p => p a b -> p s t`.
The constraint determines what the optic *can do*:

| Optic | Profunctor constraint | Tambara action | Can get | Can set | Can traverse |
|-------|----------------------|----------------|---------|---------|-------------|
| `Iso` | `Profunctor` | any | ✅ | ✅ | ✅ |
| `Lens` | `Strong` | `(a, c)` | ✅ | ✅ | ❌ multi |
| `Prism` | `Choice` | `Either a c` | ✅ partial | ✅ | ❌ multi |
| `Grate` | `Closed` | `x -> a` | ❌ direct | ✅ | ✅ zip |
| `Traversal` | `Wander` | `f a` (Applicative) | ✅ | ✅ | ✅ |
| `AffineTraversal` | `Strong + Choice` | `Maybe (a, c)` | ✅ partial | ✅ | ❌ multi |
| `Getter` | `(Strong +) Bicontravariant` | read-only | ✅ | ❌ | ❌ |
| `Setter` | `Mapping` | write-only | ❌ | ✅ | ✅ |
| `Fold` | `Bicontravariant` | fold-only | ✅ fold | ❌ | ❌ |

The hierarchy: `Iso` is the most powerful (no constraint), `Fold`/`Setter` are the weakest.
Every more-constrained optic is a *special case* of a less-constrained one.

![Full optics hierarchy](../diagrams/optics/op_optics_full.svg)

In [17]:
:set -XRankNTypes

newtype ConstApp r a = ConstApp { getConstApp :: r }
instance Functor (ConstApp r) where fmap _ (ConstApp r) = ConstApp r
instance Monoid r => Applicative (ConstApp r) where
  pure _ = ConstApp mempty
  ConstApp r1 <*> ConstApp r2 = ConstApp (r1 <> r2)

type Traversal' s a = forall f. Applicative f => (a -> f a) -> s -> f s

-- Fold helpers (all use Traversal via Const with Monoid)
toListOf :: Traversal' s a -> s -> [a]
toListOf t s = getConstApp (t (\a -> ConstApp [a]) s)


-- Manual sum implementation
sumOf' :: Traversal' s Int -> s -> Int
sumOf' t s = sum (toListOf t s)

lengthOf :: Traversal' s a -> s -> Int
lengthOf t s = length (toListOf t s)

anyOf :: Traversal' s a -> (a -> Bool) -> s -> Bool
anyOf t p s = any p (toListOf t s)

allOf :: Traversal' s a -> (a -> Bool) -> s -> Bool
allOf t p s = all p (toListOf t s)

-- Example: traversal over all Ints in a tree
data Tree a = Leaf | Node (Tree a) a (Tree a) deriving Show

nodeVals :: Traversal' (Tree a) a
nodeVals _ Leaf         = pure Leaf
nodeVals f (Node l x r) = Node <$> nodeVals f l <*> f x <*> nodeVals f r

t :: Tree Int
t = Node (Node Leaf 1 Leaf) 2 (Node Leaf 3 Leaf)

putStrLn $ "Tree: " ++ show t
putStrLn $ "toListOf nodeVals = " ++ show (toListOf nodeVals t)
putStrLn $ "sumOf' nodeVals = " ++ show (sumOf' nodeVals t)
putStrLn $ "lengthOf nodeVals = " ++ show (lengthOf nodeVals t)
putStrLn $ "anyOf nodeVals (>2) = " ++ show (anyOf nodeVals (>2) t)
putStrLn $ "allOf nodeVals (>0) = " ++ show (allOf nodeVals (>0) t)

Tree: Node (Node Leaf 1 Leaf) 2 (Node Leaf 3 Leaf)

toListOf nodeVals = [1,2,3]

sumOf' nodeVals = 6

lengthOf nodeVals = 3

anyOf nodeVals (>2) = True

allOf nodeVals (>0) = True

## 1️⃣5️⃣ Связь: Lens и комонада Store

Линза — это в точности **коалгебра комонады `Store`**: отображение `s -> Store a s`,
где `Store a s = (a -> s, a)` несёт «текущий фокус» и «способ перезаписать». Законы
комонады `Store` соответствуют законам линзы один-в-один. А сама `Store` возникает из
сопряжения `Store ⊣ State` (см. [Comonads.ipynb](Comonads.ipynb)) — поэтому линзы,
комонады и сопряжения — три грани одной конструкции.


In [18]:
-- S15: lens as a Store-comonad coalgebra
data Store s a = Store (s -> a) s
posS :: Store s a -> s
posS (Store _ s) = s
peekS :: Store s a -> s -> a
peekS (Store f _) = f

data CLens2 s a = CLens2 { lget :: s -> a, lset :: s -> a -> s }

-- a lens IS a coalgebra  s -> Store a s
lensToStore :: CLens2 s a -> s -> Store a s
lensToStore l s = Store (lset l s) (lget l s)

demoStore :: IO ()
demoStore = do
  let l  = CLens2 fst (\(_,b) a -> (a,b)) :: CLens2 (Int,Char) Int
      st = lensToStore l (1,'x')
  print (posS st)        -- 1          (= get)
  print (peekS st 99)    -- (99,'x')   (= set 99)
demoStore

1
(99,'x')

## 🔟 Оптики и `Traversable`

Связь между оптиками и `Traversable` из `FoldableTraversable.ipynb`:

```haskell
-- traverse — это в точности Traversal!
traverse :: (Traversable t, Applicative f) => (a -> f b) -> t a -> f (t b)
--                                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
--                                            это тип Traversal' (t a) a
```

Каждый `Traversable` даёт `Traversal` бесплатно.

### Иерархия через `Applicative` / `Functor`

| Оптика | Требуемый эффект | Пример |
|--------|-----------------|--------|
| Setter | `Identity` | `over lens (+1)` |
| Traversal | `Applicative f` | `traverse` для списков |
| Fold | `Applicative + Contravariant` | `toListOf` |
| Getter | `Functor + Contravariant` | `view` |
| Lens | `Functor` | `get/set` |
| Iso | `Profunctor` | `from/to` |

## 8️⃣ Библиотека `lens`

Библиотека `lens` от Edward Kmett — наиболее полная реализация оптик.

### Псевдонимы типов в `lens`

```haskell
type Lens      s t a b = forall f. Functor     f => (a->f b) -> s->f t
type Traversal s t a b = forall f. Applicative f => (a->f b) -> s->f t
type Fold s a          = forall f. (Applicative f, Contravariant f) => (a->f a) -> s->f s
type Getter s a        = forall f. (Functor f, Contravariant f)     => (a->f a) -> s->f s
type Setter s t a b    = (a -> Identity b) -> s -> Identity t
type Prism s t a b     = forall p f. (Choice p, Applicative f) => p a (f b) -> p s (f t)
type Iso s t a b       = forall p f. (Profunctor p, Functor f) => p a (f b) -> p s (f t)
```

### Ключевые функции

```haskell
-- Конструкторы
lens  :: (s->a) -> (s->b->t) -> Lens s t a b
prism :: (b->t) -> (s->Either t a) -> Prism s t a b
iso   :: (s->a) -> (b->t) -> Iso s t a b

-- Операции
view    :: Getting a s a -> s -> a          -- (^.)
set     :: ASetter s t a b -> b -> s->t     -- (.~)
over    :: ASetter s t a b -> (a->b)->s->t  -- (%~)
preview :: Getting (First a) s a -> s -> Maybe a  -- (^?)
toListOf :: Getting (Endo [a]) s a -> s -> [a]    -- (^..)

-- Операторы (инфиксные)
s ^. l       -- view
s & l .~ b   -- set
s & l %~ f   -- over
s ^? l       -- preview (Maybe)
s ^.. l      -- toListOf
```

### Пакет `optics` (современная альтернатива)

Пакет `optics` использует подход на основе профункторов с классами типов, что даёт лучшие сообщения об ошибках и большую однородность.

## 1️⃣8️⃣ Практический пример: глубокий апдейт вложенной записи

Классическая боль Haskell — обновить поле глубоко внутри вложенных записей: руками это
гнездо `r { a = (a r) { b = ... } }`. Композиция линз превращает это в **одну строку**:
`over (deptL . headL . salaryL) (+15000) company`.


In [19]:
-- S18: nested record update via composed VL lenses
:set -XRankNTypes
newtype Id3 a = Id3 { runId3 :: a }
instance Functor Id3 where fmap f (Id3 a) = Id3 (f a)
newtype Const3 r a = Const3 { getConst3 :: r }
instance Functor (Const3 r) where fmap _ (Const3 r) = Const3 r

type Lens3 s a = forall f. Functor f => (a -> f a) -> s -> f s
view3 :: Lens3 s a -> s -> a
view3 l s = getConst3 (l Const3 s)
over3 :: Lens3 s a -> (a -> a) -> s -> s
over3 l f s = runId3 (l (Id3 . f) s)

data Employee = Employee { _salary :: Int }   deriving Show
data Dept     = Dept     { _head   :: Employee } deriving Show
data Company  = Company  { _dept   :: Dept }   deriving Show

salaryL :: Lens3 Employee Int
salaryL f (Employee s) = fmap Employee (f s)
headL :: Lens3 Dept Employee
headL f (Dept h) = fmap Dept (f h)
deptL :: Lens3 Company Dept
deptL f (Company d) = fmap Company (f d)

demoNested :: IO ()
demoNested = do
  let c    = Company (Dept (Employee 100000))
      path = deptL . headL . salaryL
  print (view3 path c)            -- 100000
  print (over3 path (+15000) c)   -- salary raised, structure preserved
demoNested

100000
Company {_dept = Dept {_head = Employee {_salary = 115000}}}

## 1️⃣9️⃣ Сводка: иерархия оптик

Оптики образуют **решётку по мощности**: чем выше, тем больше можно (читать и писать,
произвольное число целей), чем ниже — тем специализированнее.

![Иерархия оптик](../diagrams/optics/op_hierarchy_poster.svg)

| Оптика | Фокус | Чтение | Запись | Целей | Кодировка (VL/проф.) |
|---|---|---|---|---|---|
| **Iso** | `s` &#8596; `a` | да | да | 1 (обратимо) | `Exchange` / profunctor |
| **Lens** | поле произведения | да | да | ровно 1 | `Strong` |
| **Prism** | ветвь суммы | preview | review | 0 или 1 | `Choice` |
| **AffineTraversal** | 0-или-1 в любом `s` | да (Maybe) | да | 0 или 1 | `Strong`+`Choice` |
| **Traversal** | много значений | да | да | 0..n | `Applicative` |
| **Fold** | много (read-only) | да | нет | 0..n | `Const` |
| **Getter** | одно (read-only) | да | нет | 1 | `Const` |
| **Setter** | модификация | нет | да | 0..n | `Identity` |
| **Grate** | коэкспоненциал | da | да | n | `Closed` |

### Единая картина

Все оптики — это **естественные преобразования** `forall p. C p => p a b -> p s t`,
где класс `C` (Strong, Choice, Applicative, Closed, ...) задаёт *вид* оптики. Поэтому
они компонуются обычной `(.)`, а профункторное представление объединяет их все под одной
сигнатурой. Решётка мощности и профункторная унификация — два взгляда на одно и то же.


---

**← Предыдущий:** [Профункторы](Profunctors.ipynb)  |  **Следующий →** [Лемма Ёнеды](YonedaLemma.ipynb)
